# 4.3 Task C — NLP Sentiment Classification\n**Autore:** Studente 3\n**Input:** `outputs/reviews_clean.parquet`, `outputs/listings_clean.parquet`\n**Output:** `models/sentiment_model.pkl`, `outputs/comment_sentiments.csv`, `metrics/sentiment_metrics.json`

In [ ]:
import pandas as pd\nimport numpy as np\nimport json\nimport joblib\nimport re\nimport os\n\nfrom sklearn.model_selection import train_test_split, StratifiedShuffleSplit, GridSearchCV\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import LinearSVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import (\n    accuracy_score, f1_score, confusion_matrix,\n    classification_report\n)

## 4.3.1 Caricamento e Join

In [ ]:
reviews = pd.read_parquet("outputs/reviews_clean.parquet")\nlistings = pd.read_parquet("outputs/listings_clean.parquet")[["id", "review_scores_rating"]]\n\n# Join per ottenere il rating del listing su ogni recensione\ndf = reviews.merge(listings, left_on="listing_id", right_on="id", how="inner")\ndf = df.dropna(subset=["comments", "review_scores_rating"])\nprint("Shape dopo join:", df.shape)

## 4.3.2 Creazione Label Sentiment

In [ ]:
def rating_to_sentiment(r):\n    if r >= 4.6:\n        return "positivo"\n    elif r >= 3.5:\n        return "neutro"\n    else:\n        return "negativo"\n\ndf["sentiment"] = df["review_scores_rating"].apply(rating_to_sentiment)\nprint(df["sentiment"].value_counts())

## 4.3.3 Preprocessing Testo

In [ ]:
# TODO: stopwords it+en, rimozione punteggiatura, eventuale langdetect\n\ndef clean_text(text):\n    if pd.isna(text):\n        return ""\n    text = str(text).lower()\n    text = re.sub(r"[^a-zàèéìòù\s]", " ", text)\n    return " ".join(text.split())\n\ndf["comments_clean"] = df["comments"].apply(clean_text)

## 4.3.4 Split Stratificato

In [ ]:
X = df["comments_clean"]\ny = df["sentiment"]\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42, stratify=y\n)

## 4.3.5 Helper Metriche

In [ ]:
def clf_metrics(name, model, X_test, y_test):\n    y_pred = model.predict(X_test)\n    acc = accuracy_score(y_test, y_pred)\n    f1 = f1_score(y_test, y_pred, average="macro")\n    print(f"{name}: Accuracy={acc:.4f}, F1-macro={f1:.4f}")\n    print(classification_report(y_test, y_pred))\n    return {"model": name, "accuracy": acc, "f1_macro": f1}

## 4.3.6 Modello 1 — Naive Bayes

In [ ]:
nb = Pipeline([\n    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),\n    ("clf", MultinomialNB())\n])\nnb.fit(X_train, y_train)\nmetrics_nb = clf_metrics("NaiveBayes", nb, X_test, y_test)

## 4.3.7 Modello 2 — Logistic Regression

In [ ]:
logreg = Pipeline([\n    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),\n    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))\n])\nlogreg.fit(X_train, y_train)\nmetrics_logreg = clf_metrics("LogisticRegression", logreg, X_test, y_test)

## 4.3.8 Modello 3 — Linear SVM

In [ ]:
svm = Pipeline([\n    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),\n    ("clf", LinearSVC(class_weight="balanced", max_iter=5000))\n])\nsvm.fit(X_train, y_train)\nmetrics_svm = clf_metrics("LinearSVC", svm, X_test, y_test)

## 4.3.9 Confronto

In [ ]:
results = [metrics_nb, metrics_logreg, metrics_svm]\npd.DataFrame(results)

## 5.3 Ottimizzazione TF-IDF + Classificatore

In [ ]:
param_grid = {\n    "tfidf__max_features": [5000, 10000, 20000],\n    "tfidf__ngram_range": [(1, 1), (1, 2)],\n    "clf__C": [0.1, 1, 10],\n}\n\ngrid = GridSearchCV(\n    logreg, param_grid=param_grid,\n    cv=3, scoring="f1_macro", n_jobs=-1\n)\ngrid.fit(X_train, y_train)\n\nprint("Best params:", grid.best_params_)\nmetrics_best = clf_metrics("LogReg_Opt", grid.best_estimator_, X_test, y_test)

## 4.3.10 Export Sentiment Score per Bridge

In [ ]:
# Predici su tutto il dataset per generare il bridge\nbest_clf = grid.best_estimator_\ndf["predicted_sentiment"] = best_clf.predict(df["comments_clean"])\n\n# Mappa sentiment -> score numerico 0–1\nscore_map = {"negativo": 0.0, "neutro": 0.5, "positivo": 1.0}\ndf["sentiment_score"] = df["predicted_sentiment"].map(score_map)\n\nbridge_df = df[["listing_id", "id_x", "predicted_sentiment", "sentiment_score"]].rename(\n    columns={"id_x": "comment_id"}\n)\nbridge_df.to_csv("outputs/comment_sentiments.csv", index=False)\nprint("Bridge exportato: outputs/comment_sentiments.csv")

## Export Modello e Metriche

In [ ]:
joblib.dump(best_clf, "models/sentiment_model.pkl")\n\nwith open("metrics/sentiment_metrics.json", "w") as f:\n    json.dump({\n        "results": results,\n        "best_model": metrics_best,\n        "best_params": grid.best_params_\n    }, f, indent=2)\n\nprint("Task C completato e esportato.")